### Comparaison des Modélisations pour la Fréquence et les Coûts des Sinistres

Dans cette section, nous allons comparer différentes approches de modélisation afin d'évaluer deux aspects clés des sinistres :

1. **La fréquence des sinistres** : Modélisation du nombre de sinistres survenus pour chaque contrat.
2. **Les coûts des sinistres** : Estimation des montants associés aux sinistres.

L'objectif est d'identifier les modèles les plus performants pour chaque aspect, en utilisant des métriques d'évaluation adaptées.

In [ ]:
#pip install statsmodels

  Using cached statsmodels-0.14.5-cp313-cp313-win_amd64.whl.metadata (9.8 kB)
  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.6 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/9.6 MB 4.2 MB/s eta 0:00:02
   --------- ------------------------------ 2.4/9.6 MB 4.6 MB/s eta 0:00:02
   -------------- ------------------------- 3.4/9.6 MB 4.7 MB/s eta 0:00:02
   ----------------- ---------------------- 4.2/9.6 MB 4.5 MB/s eta 0:00:02
   -------------------- ------------------- 5.0/9.6 MB 4.2 MB/s eta 0:00:02
   --------------------- ------------------ 5.2/9.6 MB 4.2 MB/


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import numpy as np
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error
from statsmodels.genmod.families.links import log, identity

In [2]:
freq = pd.read_parquet("data/raw/freMTPLfreq.parquet")
sev = pd.read_parquet("data/raw/freMTPLsev.parquet")

freq.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 413169 entries, 0 to 413168
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   PolicyID   413169 non-null  object 
 1   ClaimNb    413169 non-null  int32  
 2   Exposure   413169 non-null  float64
 3   Power      413169 non-null  object 
 4   CarAge     413169 non-null  int32  
 5   DriverAge  413169 non-null  int32  
 6   Brand      413169 non-null  object 
 7   Gas        413169 non-null  object 
 8   Region     413169 non-null  object 
 9   Density    413169 non-null  int32  
dtypes: float64(1), int32(4), object(5)
memory usage: 25.2+ MB


In [17]:
X=freq[['Exposure', 'CarAge', 'DriverAge',
       'Brand', 'Gas', 'Region', 'Density']]
y=freq['ClaimNb']

# Encodage des variables catégorielles
X = pd.get_dummies(X, drop_first=True, dtype=int)
display(X)

# Split 75% / 25%
X_train, X_test, y_train, y_test_ClaimN = train_test_split(X, y, test_size=0.25, random_state=42)


,Exposure,CarAge,DriverAge,Density,Brand_Japanese (except Nissan) or Korean,"Brand_Mercedes, Chrysler or BMW","Brand_Opel, General Motors or Ford","Brand_Renault, Nissan or Citroen","Brand_Volkswagen, Audi, Skoda or Seat",Brand_other,Gas_Regular,Region_Basse-Normandie,Region_Bretagne,Region_Centre,Region_Haute-Normandie,Region_Ile-de-France,Region_Limousin,Region_Nord-Pas-de-Calais,Region_Pays-de-la-Loire,Region_Poitou-Charentes
0,0.090000,0,46,76,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0.840000,0,46,76,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0.520000,2,38,3003,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0
3,0.450000,2,38,3003,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0
4,0.150000,0,41,60,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413164,0.002740,0,29,2471,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
413165,0.005479,0,29,5360,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0
413166,0.005479,0,49,5360,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
413167,0.002740,0,41,9850,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0




#### Principe de la fonction
La fonction `stepwise_selection` est utilisée pour effectuer une sélection de variables explicatives dans un modèle statistique, en se basant sur des critères d'information comme l'AIC (Akaike Information Criterion) ou le BIC (Bayesian Information Criterion). Voici les étapes principales de cette fonction :

1. **Initialisation** :
    - Toutes les variables explicatives présentes dans `X` sont incluses dans le modèle initial.
    - Un modèle est ajusté avec toutes les variables, et la valeur du critère choisi (AIC ou BIC) est calculée.

2. **Itérations** :
    - À chaque itération, la fonction teste le retrait de chaque variable explicative une par une.
    - Pour chaque sous-ensemble de variables (obtenu en retirant une variable), un modèle est ajusté, et la valeur du critère est calculée.

3. **Sélection de la variable à retirer** :
    - La variable dont le retrait entraîne la plus faible valeur du critère est identifiée.
    - Si cette nouvelle valeur est inférieure à celle du modèle actuel, la variable est retirée du modèle, et le processus continue.
    - Sinon, l'algorithme s'arrête, car retirer d'autres variables n'améliore plus le critère.

4. **Résultat final** :
    - Le modèle final est ajusté avec les variables sélectionnées.
    - La fonction retourne le modèle final et la liste des variables sélectionnées.


In [18]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 309876 entries, 408079 to 121958
Data columns (total 20 columns):
 #   Column                                    Non-Null Count   Dtype  
---  ------                                    --------------   -----  
 0   Exposure                                  309876 non-null  float64
 1   CarAge                                    309876 non-null  int32  
 2   DriverAge                                 309876 non-null  int32  
 3   Density                                   309876 non-null  int32  
 4   Brand_Japanese (except Nissan) or Korean  309876 non-null  int64  
 5   Brand_Mercedes, Chrysler or BMW           309876 non-null  int64  
 6   Brand_Opel, General Motors or Ford        309876 non-null  int64  
 7   Brand_Renault, Nissan or Citroen          309876 non-null  int64  
 8   Brand_Volkswagen, Audi, Skoda or Seat     309876 non-null  int64  
 9   Brand_other                               309876 non-null  int64  
 10  Gas_Regular         

In [23]:
# Fonction pour effectuer la sélection de variables basée sur AIC et BIC
def stepwise_selection(X, y, family, criterion='AIC'):
    """
    Effectue une sélection de variables en utilisant AIC ou BIC.
    
    Parameters:
        X (pd.DataFrame): Variables explicatives.
        y (pd.Series): Variable cible.
        family: Famille de distribution (ex: sm.families.Poisson()).
        criterion (str): Critère de sélection ('AIC' ou 'BIC').
    
    Returns:
        result (GLMResultsWrapper): Résultat du modèle final.
        selected_features (list): Liste des variables sélectionnées.
    """
    selected_features = list(X.columns)
    current_model = sm.GLM(y, sm.add_constant(X[selected_features]), family=family).fit()
    current_criterion = getattr(current_model, criterion.lower())
    
    while True:
        criteria = []
        for feature in selected_features:
            features_to_test = [f for f in selected_features if f != feature]
            model = sm.GLM(y, sm.add_constant(X[features_to_test]), family=family).fit()
            model_criterion = getattr(model, criterion.lower())
            criteria.append((model_criterion, feature))
            
            # Afficher la valeur du critère pour chaque modèle
            print(f"Modèle'{features_to_test}': {criterion} = {model_criterion}")
        
        criteria.sort()
        best_criterion, worst_feature = criteria[0]
        
        if best_criterion < current_criterion:
            selected_features.remove(worst_feature)
            current_criterion = best_criterion
        else:
            break
    
    final_model = sm.GLM(y, sm.add_constant(X[selected_features]), family=family).fit()
    return final_model, selected_features



In [20]:


# Appliquer la sélection de variables
final_model_AIC_ClaimN, selected_features_AIC_ClaimN = stepwise_selection(X_train, y_train, sm.families.Poisson(), criterion='AIC')

# Afficher les résultats
print("Variables sélectionnées :", selected_features_AIC_ClaimN)
display(final_model_AIC_ClaimN.summary())

KeyboardInterrupt: 

In [9]:
import warnings

# Appliquer la sélection de variables
warnings.filterwarnings("ignore")

final_model_BIC_ClaimN, selected_features_BIC_ClaimN = stepwise_selection(X_train, y_train, sm.families.Poisson(), criterion='BIC')

# Afficher les résultats
print("Variables sélectionnées :", selected_features_BIC_ClaimN)
display(final_model_BIC_ClaimN.summary())

Variables sélectionnées : ['Exposure', 'CarAge', 'DriverAge', 'Density', 'Brand_Japanese (except Nissan) or Korean', 'Brand_Renault, Nissan or Citroen', 'Gas_Regular']


<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                ClaimNb   No. Observations:               309876
Model:                            GLM   Df Residuals:                   309868
Model Family:                 Poisson   Df Model:                            7
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -50453.
Date:                Wed, 03 Dec 2025   Deviance:                       77578.
Time:                        10:18:37   Pearson chi2:                 3.25e+05
No. Iterations:                     7   Pseudo R-squ. (CS):           0.007796
Covariance Type:            nonrobust                                         
============================================================================================================
                                               coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------
const                                       -3.4690      0.038    -90.315      0.000      -3.544      -3.394
Exposure                                     1.2073      0.028     42.882      0.000       1.152       1.262
CarAge                                      -0.0093      0.002     -5.186      0.000      -0.013      -0.006
DriverAge                                   -0.0076      0.001    -11.541      0.000      -0.009      -0.006
Density                                   1.857e-05   1.81e-06     10.247      0.000     1.5e-05    2.21e-05
Brand_Japanese (except Nissan) or Korean    -0.3525      0.032    -10.904      0.000      -0.416      -0.289
Brand_Renault, Nissan or Citroen            -0.1093      0.020     -5.360      0.000      -0.149      -0.069
Gas_Regular                                 -0.1067      0.019     -5.700      0.000      -0.143      -0.070
============================================================================================================
"""

In [10]:
selected_features_AIC_ClaimN

['Exposure',
 'CarAge',
 'DriverAge',
 'Density',
 'Brand_Japanese (except Nissan) or Korean',
 'Brand_Opel, General Motors or Ford',
 'Brand_Renault, Nissan or Citroen',
 'Gas_Regular',
 'Region_Bretagne',
 'Region_Haute-Normandie',
 'Region_Ile-de-France',
 'Region_Limousin',
 'Region_Nord-Pas-de-Calais',
 'Region_Pays-de-la-Loire',
 'Region_Poitou-Charentes']

In [11]:
selected_features_BIC_ClaimN

['Exposure',
 'CarAge',
 'DriverAge',
 'Density',
 'Brand_Japanese (except Nissan) or Korean',
 'Brand_Renault, Nissan or Citroen',
 'Gas_Regular']

In [12]:
# Ajouter une constante à X_test pour les deux modèles
X_test_const_AIC_ClaimN = sm.add_constant(X_test[selected_features_AIC_ClaimN])
X_test_const_BIC_ClaimN = sm.add_constant(X_test[selected_features_BIC_ClaimN])

# Prédictions pour les deux modèles
y_pred_test_AIC_ClaimN = final_model_AIC_ClaimN.predict(X_test_const_AIC_ClaimN)
y_pred_test_BIC_ClaimN = final_model_BIC_ClaimN.predict(X_test_const_BIC_ClaimN)

# Calcul des MSE pour les deux modèles
mse_AIC = mean_squared_error(y_test_ClaimN, y_pred_test_AIC_ClaimN)
mse_BIC = mean_squared_error(y_test_ClaimN, y_pred_test_BIC_ClaimN)

print("MSE pour le modèle AIC ClaimN :", mse_AIC)
print("MSE pour le modèle BIC ClaimN :", mse_BIC)


MSE pour le modèle AIC ClaimN : 0.04211466893706511
MSE pour le modèle BIC ClaimN : 0.04211065684750139


In [13]:
y_pred_test_AIC_ClaimN.mean()

np.float64(0.03901465961897154)

In [14]:

y_pred_test_BIC_ClaimN.mean()

np.float64(0.03900624982056504)

In [15]:
y_test_ClaimN.mean()

np.float64(0.03978004317814373)

In [16]:
count_claim_nb_ge_1 = (y_test_ClaimN >= 1).sum()
print("Nombre de ClaimNb >= 1 :", count_claim_nb_ge_1)

Nombre de ClaimNb >= 1 : 3902


## Coût des sinistres

In [10]:
#Combiner les 2 bases de données

freq['PolicyID'] = freq['PolicyID'].astype(str)
sev['PolicyID'] = sev['PolicyID'].astype(str)

# Agrégation de sev : un contrat peut avoir plusieurs sinistres
sev_agg = (
    sev.groupby("PolicyID")
       .agg(
           ClaimAmount=("ClaimAmount", "sum"),   # coût total des sinistres du contrat
           ClaimNb_sev=("ClaimAmount", "size")   # nombre de sinistres déclarés dans sev
       )
       .reset_index()
)

print(len(sev), "lignes sev brutes")
print(len(sev_agg), "lignes sev agrégées (1 par PolicyID)")

merged = pd.merge(freq, sev_agg, on="PolicyID", how="left")

# Remplacer les NaN (contrats sans sinistre) par 0
merged["ClaimAmount"] = merged["ClaimAmount"].fillna(0)
merged["ClaimNb_sev"] = merged["ClaimNb_sev"].fillna(0)

print("Nb de lignes freq :", len(freq))
print("Nb de lignes merged :", len(merged))


16181 lignes sev brutes
15390 lignes sev agrégées (1 par PolicyID)
Nb de lignes freq : 413169
Nb de lignes merged : 413169


In [ ]:
display(merged)


,PolicyID,ClaimNb,Exposure,Power,CarAge,DriverAge,Brand,Gas,Region,Density,ClaimAmount,ClaimNb_sev
0,1,0,0.090000,g,0,46,Japanese (except Nissan) or Korean,Diesel,Aquitaine,76,0.0,0.0
1,2,0,0.840000,g,0,46,Japanese (except Nissan) or Korean,Diesel,Aquitaine,76,0.0,0.0
2,3,0,0.520000,f,2,38,Japanese (except Nissan) or Korean,Regular,Nord-Pas-de-Calais,3003,0.0,0.0
3,4,0,0.450000,f,2,38,Japanese (except Nissan) or Korean,Regular,Nord-Pas-de-Calais,3003,0.0,0.0
4,5,0,0.150000,g,0,41,Japanese (except Nissan) or Korean,Diesel,Pays-de-la-Loire,60,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
413164,413165,0,0.002740,j,0,29,Japanese (except Nissan) or Korean,Diesel,Ile-de-France,2471,0.0,0.0
413165,413166,0,0.005479,d,0,29,Japanese (except Nissan) or Korean,Regular,Ile-de-France,5360,0.0,0.0
413166,413167,0,0.005479,k,0,49,Japanese (except Nissan) or Korean,Diesel,Ile-de-France,5360,0.0,0.0
413167,413168,0,0.002740,d,0,41,Japanese (except Nissan) or Korean,Regular,Ile-de-France,9850,0.0,0.0


In [64]:
X=merged[['Exposure', 'CarAge', 'DriverAge',
       'Brand', 'Gas', 'Region', 'Density', 'ClaimNb']]
y=merged['ClaimAmount']

# Encodage des variables catégorielles
X = pd.get_dummies(X, drop_first=True, dtype=int)
display(X)

# Split 75% / 25%
X_train, X_test, y_train, y_test_ClaimA = train_test_split(X, y, test_size=0.25, random_state=42)

,Exposure,CarAge,DriverAge,Density,ClaimNb,Brand_Japanese (except Nissan) or Korean,"Brand_Mercedes, Chrysler or BMW","Brand_Opel, General Motors or Ford","Brand_Renault, Nissan or Citroen","Brand_Volkswagen, Audi, Skoda or Seat",...,Gas_Regular,Region_Basse-Normandie,Region_Bretagne,Region_Centre,Region_Haute-Normandie,Region_Ile-de-France,Region_Limousin,Region_Nord-Pas-de-Calais,Region_Pays-de-la-Loire,Region_Poitou-Charentes
0,0.090000,0,46,76,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0.840000,0,46,76,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0.520000,2,38,3003,0,1,0,0,0,0,...,1,0,0,0,0,0,0,1,0,0
3,0.450000,2,38,3003,0,1,0,0,0,0,...,1,0,0,0,0,0,0,1,0,0
4,0.150000,0,41,60,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413164,0.002740,0,29,2471,0,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
413165,0.005479,0,29,5360,0,1,0,0,0,0,...,1,0,0,0,0,1,0,0,0,0
413166,0.005479,0,49,5360,0,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
413167,0.002740,0,41,9850,0,1,0,0,0,0,...,1,0,0,0,0,1,0,0,0,0


In [29]:
# Vérifier les valeurs manquantes dans le DataFrame merged
missing_values = merged.isnull().sum()
print(missing_values)

PolicyID       0
ClaimNb        0
Exposure       0
Power          0
CarAge         0
DriverAge      0
Brand          0
Gas            0
Region         0
Density        0
ClaimAmount    0
ClaimNb_sev    0
dtype: int64


In [33]:
y_train.mean()

np.float64(85.99866398172172)

In [34]:
# Fit a GLM model using all features in X_train
glm_model = sm.GLM(y_train, sm.add_constant(X_train), family=sm.families.Gamma(link=log())).fit()

# Display the summary of the model
print(glm_model.summary())

c:\Users\thoma\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\genmod\families\links.py:13: FutureWarning: The log link alias is deprecated. Use Log instead. The log link alias will be removed after the 0.15.0 release.
  warnings.warn(


                 Generalized Linear Model Regression Results                  
Dep. Variable:            ClaimAmount   No. Observations:               309876
Model:                            GLM   Df Residuals:                   309854
Model Family:                   Gamma   Df Model:                           21
Link Function:                    log   Scale:                          5.3670
Method:                          IRLS   Log-Likelihood:                    inf
Date:                Wed, 03 Dec 2025   Deviance:                   2.0970e+07
Time:                        10:54:22   Pearson chi2:                 1.66e+06
No. Iterations:                   100   Pseudo R-squ. (CS):                nan
Covariance Type:            nonrobust                                         
                                               coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------

In [37]:
# Ajouter une constante à X_test pour les deux modèles
X_test_const_AIC = sm.add_constant(X_test)
#X_test_const_BIC = sm.add_constant(X_test[selected_features_BIC_ClaimA])

# Prédictions pour les deux modèles
y_pred_test = glm_model.predict(X_test_const_AIC)
#y_pred_test_BIC_ClaimA = final_model_BIC_ClaimA.predict(X_test_const_BIC)

# Calcul des MSE pour les deux modèles
mse_AIC = mean_squared_error(y_test_ClaimA, y_pred_test)
#mse_BIC = mean_squared_error(y_test_ClaimA, y_pred_test_BIC_ClaimA)
print("MSE pour le modèle AIC ClaimA :", mse_AIC)
#print("MSE pour le modèle BIC ClaimA :", mse_BIC)

MSE pour le modèle AIC ClaimA : 7.827860277932774e+125


In [54]:
y_train.max()

np.float64(2036833.0)

In [51]:
y_pred_test.max()

np.float64(2.8435245237003146e+65)

In [57]:
y_pred_test_rounded = y_pred_test.round()
print(y_pred_test_rounded)

y_pred_test_rounded[y_pred_test_rounded > y_train.max()] = 0
mse_AIC = mean_squared_error(y_test_ClaimA, y_pred_test_rounded)
print("MSE pour le modèle AIC ClaimA après arrondi et ajustement :", mse_AIC)

350286       0.0
127332       0.0
204803       0.0
286289       0.0
318780       0.0
           ...  
75272        0.0
224088       0.0
202028       0.0
48434        0.0
39113     1869.0
Length: 103293, dtype: float64
MSE pour le modèle AIC ClaimA après arrondi et ajustement : 2457361.389939299


In [61]:
y_pred_test_rounded.max()
#y_pred_test_rounded.mean()

np.float64(1901.0)

In [63]:
y_train[y_train > 0].mean()

np.float64(2319.7181406685236)

In [ ]:
y_pred_test_rounded

In [60]:
y_test_ClaimA.mean()

np.float64(75.6697452876768)

In [53]:
y_pred_test_rounded[24877]

np.float64(9.90773521286626e+23)

In [52]:
y_test_ClaimA[24877]

np.float64(1212.0)

In [24]:
# Appliquer la sélection de variables
final_model_AIC_ClaimA, selected_features_AIC_ClaimA = stepwise_selection(X_train, y_train, sm.families.Gamma(link=log()), criterion='AIC')

# Afficher les résultats
print("Variables sélectionnées :", selected_features_AIC_ClaimA)
display(final_model_AIC_ClaimA.summary())

c:\Users\thoma\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\genmod\families\links.py:13: FutureWarning: The log link alias is deprecated. Use Log instead. The log link alias will be removed after the 0.15.0 release.
  warnings.warn(


Modèle'['CarAge', 'DriverAge', 'Density', 'Brand_Japanese (except Nissan) or Korean', 'Brand_Mercedes, Chrysler or BMW', 'Brand_Opel, General Motors or Ford', 'Brand_Renault, Nissan or Citroen', 'Brand_Volkswagen, Audi, Skoda or Seat', 'Brand_other', 'Gas_Regular', 'Region_Basse-Normandie', 'Region_Bretagne', 'Region_Centre', 'Region_Haute-Normandie', 'Region_Ile-de-France', 'Region_Limousin', 'Region_Nord-Pas-de-Calais', 'Region_Pays-de-la-Loire', 'Region_Poitou-Charentes']': AIC = -inf
Modèle'['Exposure', 'DriverAge', 'Density', 'Brand_Japanese (except Nissan) or Korean', 'Brand_Mercedes, Chrysler or BMW', 'Brand_Opel, General Motors or Ford', 'Brand_Renault, Nissan or Citroen', 'Brand_Volkswagen, Audi, Skoda or Seat', 'Brand_other', 'Gas_Regular', 'Region_Basse-Normandie', 'Region_Bretagne', 'Region_Centre', 'Region_Haute-Normandie', 'Region_Ile-de-France', 'Region_Limousin', 'Region_Nord-Pas-de-Calais', 'Region_Pays-de-la-Loire', 'Region_Poitou-Charentes']': AIC = -inf
Modèle'['Exp

KeyboardInterrupt: 

In [16]:
import warnings

# Appliquer la sélection de variables
warnings.filterwarnings("ignore")

final_model_BIC_ClaimA, selected_features_BIC_ClaimA = stepwise_selection(X_train, y_train, sm.families.Gamma(), criterion='BIC')

# Afficher les résultats
print("Variables sélectionnées :", selected_features_BIC_ClaimA)
display(final_model_BIC_ClaimA.summary())

Variables sélectionnées : ['Density', 'Brand_Opel, General Motors or Ford', 'Gas_Regular', 'Region_Centre', 'Region_Limousin', 'Region_Nord-Pas-de-Calais', 'Region_Pays-de-la-Loire', 'Region_Poitou-Charentes']


<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:            ClaimAmount   No. Observations:               310470
Model:                            GLM   Df Residuals:                   310461
Model Family:                   Gamma   Df Model:                            8
Link Function:           InversePower   Scale:                          1550.7
Method:                          IRLS   Log-Likelihood:                    inf
Date:                Mon, 01 Dec 2025   Deviance:                   2.1450e+07
Time:                        22:18:36   Pearson chi2:                 4.81e+08
No. Iterations:                    19   Pseudo R-squ. (CS):                nan
Covariance Type:            nonrobust                                         
======================================================================================================
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
const                                  0.0143      0.002      7.286      0.000       0.010       0.018
Density                             9.585e-08   2.31e-07      0.415      0.678   -3.57e-07    5.49e-07
Brand_Opel, General Motors or Ford     0.0013      0.003      0.420      0.674      -0.005       0.007
Gas_Regular                           -0.0022      0.002     -1.356      0.175      -0.005       0.001
Region_Centre                         -0.0044      0.002     -2.269      0.023      -0.008      -0.001
Region_Limousin                        0.0050      0.012      0.398      0.690      -0.019       0.029
Region_Nord-Pas-de-Calais              0.0039      0.005      0.772      0.440      -0.006       0.014
Region_Pays-de-la-Loire                0.0025      0.004      0.618      0.537      -0.005       0.010
Region_Poitou-Charentes                0.0019      0.005      0.356      0.722      -0.008       0.012
======================================================================================================
"""

In [17]:
selected_features_AIC_ClaimA

['Exposure',
 'CarAge',
 'DriverAge',
 'Density',
 'Brand_Japanese (except Nissan) or Korean',
 'Brand_Mercedes, Chrysler or BMW',
 'Brand_Opel, General Motors or Ford',
 'Brand_other',
 'Brand_Renault, Nissan or Citroen',
 'Brand_Volkswagen, Audi, Skoda or Seat',
 'Gas_Regular',
 'Region_Basse-Normandie',
 'Region_Bretagne',
 'Region_Centre',
 'Region_Haute-Normandie',
 'Region_Ile-de-France',
 'Region_Limousin',
 'Region_Nord-Pas-de-Calais',
 'Region_Pays-de-la-Loire',
 'Region_Poitou-Charentes']

In [18]:
selected_features_BIC_ClaimA

['Density',
 'Brand_Opel, General Motors or Ford',
 'Gas_Regular',
 'Region_Centre',
 'Region_Limousin',
 'Region_Nord-Pas-de-Calais',
 'Region_Pays-de-la-Loire',
 'Region_Poitou-Charentes']

In [ ]:
# Ajouter une constante à X_test pour les deux modèles
X_test_const_AIC = sm.add_constant(X_test[selected_features_AIC_ClaimA])
#X_test_const_BIC = sm.add_constant(X_test[selected_features_BIC_ClaimA])

# Prédictions pour les deux modèles
y_pred_test_AIC_ClaimA = final_model_AIC_ClaimA.predict(X_test_const_AIC)
#y_pred_test_BIC_ClaimA = final_model_BIC_ClaimA.predict(X_test_const_BIC)

# Calcul des MSE pour les deux modèles
mse_AIC = mean_squared_error(y_test_ClaimA, y_pred_test_AIC_ClaimA)
#mse_BIC = mean_squared_error(y_test_ClaimA, y_pred_test_BIC_ClaimA)
print("MSE pour le modèle AIC ClaimA :", mse_AIC)
#print("MSE pour le modèle BIC ClaimA :", mse_BIC)

MSE pour le modèle AIC ClaimA : 77070358.48873481
MSE pour le modèle BIC ClaimA : 1929162.0829623197


In [33]:
print(y_pred_test_AIC_ClaimA.mean())
print(y_pred_test_BIC_ClaimA.mean())
print(y_test_ClaimA.mean())

78.87475363735635
86.42535681064042
73.71916127162045


0n remarque que les prediction de coût sont bonne en moyenne mais sous estime totalement la variance des sinistre réels qui sont au nombre de 3000 environ avec un cout moyen de 1900 ----> à REVOIR 